In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split 
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score
import seaborn as sns
import matplotlib.pyplot as plt 
import requests
import zipfile 
import io

# Load and preprocess the dataset
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip'
response = requests.get(url)
with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
    zip_ref.extractall()

# Load the dataset
df = pd.read_csv('SMSSpamCollection', sep='\t', header=None, names=['label', 'message'])
# Convert labels to binary: spam -> 1, ham -> 0
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(df['message'], df['label'], test_size=0.3, random_state=42)

print("Data loaded and split successfully.")

In [ ]:
# Vectorize the text data
vectorizer = CountVectorizer(stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

alphas = [0.5, 1.0, 2.0]
for alpha in alphas:
    nb_classifier = MultinomialNB(alpha=alpha)
    nb_classifier.fit(X_train_vec, y_train)
    y_pred = nb_classifier.predict(X_test_vec)
    acc = accuracy_score(y_test, y_pred)
    print(f"Alpha = {alpha}: Accuracy = {acc:.4f}")

In [ ]:
# Using TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

nb_classifier_tfidf = MultinomialNB(alpha=1.0)
nb_classifier_tfidf.fit(X_train_tfidf, y_train)
y_pred_tfidf = nb_classifier_tfidf.predict(X_test_tfidf)

print("Accuracy with TF-IDF:", accuracy_score(y_test, y_pred_tfidf))
print("\nClassification Report with TF-IDF:\n", classification_report(y_test, y_pred_tfidf))

In [ ]:
# Perform 5-fold cross-validation
X_all_vec = vectorizer.fit_transform(df['message'])
y_all = df['label']

nb_classifier_cv = MultinomialNB(alpha=1.0)
scores = cross_val_score(nb_classifier_cv, X_all_vec, y_all, cv=5)

print("Cross-Validation Scores:", scores)
print("Mean CV Accuracy: {:.4f}".format(scores.mean()))

In [ ]:
print("Original class distribution:\n", df['label'].value_counts())

# Manual oversampling with Pandas
ham_msgs = df[df['label'] == 0]
spam_msgs = df[df['label'] == 1]

# Oversample spam to match ham count
spam_oversampled = spam_msgs.sample(len(ham_msgs), replace=True, random_state=42)
df_balanced = pd.concat([ham_msgs, spam_oversampled])

print("\nClass distribution after oversampling:\n", df_balanced['label'].value_counts())

# Retrain on balanced data
X_train_bal, X_test_bal, y_train_bal, y_test_bal = train_test_split(df_balanced['message'], df_balanced['label'], test_size=0.3, random_state=42)
vectorizer_bal = CountVectorizer(stop_words='english')
X_train_bal_vec = vectorizer_bal.fit_transform(X_train_bal)
X_test_bal_vec = vectorizer_bal.transform(X_test_bal)

nb_classifier_bal = MultinomialNB(alpha=1.0)
nb_classifier_bal.fit(X_train_bal_vec, y_train_bal)
y_pred_bal = nb_classifier_bal.predict(X_test_bal_vec)

print("\nAccuracy after balancing:", accuracy_score(y_test_bal, y_pred_bal))
print("\nClassification Report after balancing:\n", classification_report(y_test_bal, y_pred_bal))

In [ ]:
# Re-train model for final use
nb_classifier.fit(X_train_vec, y_train)

def detect_spam(message):
    message_vec = vectorizer.transform([message])
    prediction = nb_classifier.predict(message_vec)
    return "Spam" if prediction[0] == 1 else "Ham"

new_messages = [
    "Congratulations! You've won a $1,000 Walmart gift card. Go to http://bit.ly/12345 to claim now.",
    "Hey, are we still on for lunch tomorrow?",
    "URGENT! Your mobile number has been awarded with a �2000 prize GUARANTEED. Call 09061790121 from landline."
]

for msg in new_messages:
    print(f"Message: '{msg}'\nPrediction: {detect_spam(msg)}\n")